# Beta 分布的极大似然估计（解析梯度）

> 对应代码：`MathStatsCode/code_in_notes/Chap.11/MLE_beta.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：手写 Beta 分布的对数似然与解析梯度（digamma 函数），用 ml model lf1 进行极大似然估计。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.11`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.11")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear` 清空数据，`set seed 8855` 固定随机数种子。


In [ ]:
%%stata
clear
set seed 8855


**目标函数**

本段演示用**解析梯度**做极大似然估计（lf1 评估器），包含三个环节：

**① 编写似然函数**：程序 `betaobj` 的 `args todo b lnfj g1 g2` 是 lf1 评估器固定接口——`todo` 告诉程序该计算哪些内容（0=只需似然值），`b` 是参数向量，`lnfj` 是每个观测的对数似然贡献，`g1`/`g2` 是两个参数（α、β）的一阶导数列。`mleval` 从 b 中取出第1、2个方程的参数。Beta 分布的对数似然为 lnΓ(α+β)-lnΓ(α)-lnΓ(β)+(α-1)ln x+(β-1)ln(1-x)，程序用 `lngamma()` 实现；当 `todo==0` 时 `exit` 提前返回（不计算梯度以节省时间）。梯度公式用到 `digamma()`（Γ 函数的对数导数）。

**② 生成数据**：`set obs 1000`、`gen x=rbeta(0.5,3)` 模拟来自 Beta(0.5,3) 的样本。

**③ 估计**：`ml model lf1 betaobj (alpha: x=, freeparm) /beta` 声明模型——`lf1` 表示用户提供对数似然与一阶梯度（评分法）；α 写成带 `freeparm` 的方程（只估计常数项 α），β 写成 `/beta` 形式的参数；`ml search` 自动搜索初始值，`ml maximize` 执行优化。结果应接近真值 (0.5, 3)，这正是后面 MLE 数值梯度的对照版本。


In [ ]:
%%stata
cap program drop betaobj 
program betaobj 	
	args todo b lnfj g1 g2
	tempvar alpha beta lnx ln1x
	mleval `alpha'=`b', eq(1)
	mleval `beta'=`b', eq(2)
	quietly{
		gen `lnx'=log($ML_y1)
		gen `ln1x'=log(1-$ML_y1)
		replace `lnfj'=lngamma(`alpha'+`beta')-lngamma(`alpha')-lngamma(`beta')+(`alpha'-1)*`lnx'+(`beta'-1)*`ln1x'
		//如果不需要计算导数，退出
		if (`todo'==0) exit
		replace `g1'=digamma(`alpha'+`beta')-digamma(`alpha')+`lnx'
		replace `g2'=digamma(`alpha'+`beta')-digamma(`beta')+`ln1x'
	} 
end 
set obs 1000 
gen x=rbeta(0.5,3)
ml model lf1 betaobj (alpha: x = , freeparm) /beta
ml search 
ml maximize
